# 🎩 Alfred, the Gala Agent: an Agentic RAG walkthrough

This notebook follows [Hugging Face Agents Course, Unit 3](https://huggingface.co/learn/agents-course/unit3/agentic-rag/introduction). Alfred is a **smolagents `CodeAgent`** that helps run an extravagant gala. He has four tools:

| Tool | Source | Used for |
|---|---|---|
| `guest_info_retriever` | private dataset + BM25 (RAG) | who our guests are |
| `web_search` | DuckDuckGo | current public news |
| `weather_info` | simulated | planning outdoor events |
| `hub_stats` | `huggingface_hub` API | a guest org's most popular model |

A `CodeAgent` writes Python code that calls these tools as functions. It reads the results and decides whether to call more tools or give a final answer.

The code lives in `retriever.py`, `tools.py` and `agent.py`. This notebook imports from them and demos each piece.

## 1. Setup

Run this notebook with the project's `.venv` kernel (`pip install -r requirements.txt` installs `ipykernel`).

Settings come from `.env` (copy `.env.example`):
- `HF_TOKEN`: your Hugging Face token.
- `ALFRED_BACKEND`: `hf` for Hugging Face Inference Providers, or `ollama` for a free local model.

The tools in sections 2 to 5 do **not** need a model or token. Only the full agent in section 6 does.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("Backend:", os.getenv("ALFRED_BACKEND", "hf"))
print("HF token set:", bool(os.getenv("HF_TOKEN")))  # never print the token itself

## 2. The private guest RAG tool

The guest list is the Hub dataset `agents-course/unit3-invitees`. It has the columns `name`, `relation`, `description` and `email`. We **combine each guest's fields into one document** so a single search matches on any of them.

Retrieval uses **BM25**, a keyword-ranking algorithm. It needs no embedding model and works well for a short list of named people. This is the "R" in RAG: Alfred retrieves private facts the LLM could never know and uses them to ground his answer.

In [ ]:
from retriever import GuestInfoRetrieverTool, load_guest_documents

docs = load_guest_documents()
print(f"{len(docs)} guest documents. First one:\n")
print(docs[0])

In [ ]:
guest_tool = GuestInfoRetrieverTool(docs=docs)
print(guest_tool("Lady Ada Lovelace"))
print("\n--- search by relation ---")
print(guest_tool("friend from university"))

## 3. Weather tool (simulated)

No real API is called. The "forecast" is chosen from a hash of the location name, so the same city always gets the same weather. That keeps demos and tests repeatable.

In [ ]:
from tools import WeatherInfoTool

weather_tool = WeatherInfoTool()
for city in ["Paris", "London", "New York"]:
    print(weather_tool(city))

## 4. Hugging Face Hub stats tool

This tool calls `HfApi().list_models(author=..., sort="downloads", limit=1)` to find an organization's most downloaded model. Download counts are live and change daily.

In [ ]:
from tools import HubStatsTool

hub_tool = HubStatsTool()
print(hub_tool("Qwen"))
print(hub_tool("facebook"))

## 5. Web search tool

This is smolagents' built-in `DuckDuckGoSearchTool`, wrapped so that a rate limit or empty result comes back as a message rather than an exception. That way Alfred can rephrase his query instead of crashing.

In [ ]:
from tools import SafeWebSearchTool

search_tool = SafeWebSearchTool(max_results=3)
print(search_tool("wireless energy transmission latest news"))

## 6. The full agent

`build_alfred()` gives all four tools to a `CodeAgent` (plus the built-in `final_answer` tool). On each step the agent:

1. thinks about what it still needs,
2. writes a Python snippet that calls one or more tools,
3. reads the printed results (the *observation*),
4. repeats until it calls `final_answer(...)`.

`tools_used()` scans the code the agent wrote to show which tools it actually called.

⚠️ This section needs a working model: HF credits (`ALFRED_BACKEND=hf`) or a running Ollama (`ALFRED_BACKEND=ollama`).

In [ ]:
from agent import build_alfred, tools_used

alfred = build_alfred()
print("Model:", alfred.model.model_id)
print("Tools:", list(alfred.tools))

### Single tool: guest lookup (RAG)

In [ ]:
answer = alfred.run("Tell me about our guest named Lady Ada Lovelace.")
print("\n🎩 Alfred:", answer)
print("Tools used:", tools_used(alfred))

### Multi-tool: private guest info + public web search

This is the core idea of **agentic** RAG. Alfred must notice that the question needs both private facts (who Tesla is to us) and public facts (recent news), then combine them.

In [ ]:
answer = alfred.run(
    "I need to speak with Dr. Nikola Tesla about recent advancements in wireless energy. "
    "Can you help me prepare for this conversation?"
)
print("\n🎩 Alfred:", answer)
print("Tools used:", tools_used(alfred))

## 7. Chat UI

To chat with Alfred in the browser, run `python app.py` from a terminal and open http://127.0.0.1:7860. It uses smolagents' `GradioUI`, which also shows each step's code and tool output.